## Prerequisites

Before starting, ensure you have:
- [ ] AWS Account with appropriate permissions
- [ ] IAM User with CloudFormation, EC2, RDS, S3, ECS, Route53 permissions
- [ ] Git installed on your machine
- [ ] Terminal/Command Line access

## Stack Deployment Order

The following stacks must be deployed in this exact order:
1. **Networking** - VPC, Subnets, Security Groups
2. **S3** - Storage buckets
3. **SecretsManager** - Database credentials
4. **LoadBalancer** - Application Load Balancer and Target Groups
5. **RDS** - Database infrastructure
6. **Route53** - DNS routing
7. **CloudWatch** - Monitoring and logging
8. **ECS** - Container services (depends on all above)

## 1. Clone the Repository

In [ ]:
%%bash
git clone https://github.com/BAH-RADx/RADx-Replication.git
cd RADx-Replication
ls -la  # Verify files are present

## 2. Install AWS CLI

In [ ]:
%%bash
# For MacOS:
brew install awscli
# Verify installation
aws --version

## 3. Configure AWS Credentials

**Steps to get AWS credentials:**
1. Log into AWS Console
2. Go to IAM → Users → Select your user
3. Security Credentials tab → Create Access Key
4. Download and save the credentials securely

Replace `YOUR_ACCESS_KEY_ID` and `YOUR_SECRET_ACCESS_KEY` with actual values:

In [ ]:
%%bash
mkdir -p ~/.aws
cat <<EOL >> ~/.aws/credentials
[radx-rep]
aws_access_key_id=YOUR_ACCESS_KEY_ID
aws_secret_access_key=YOUR_SECRET_ACCESS_KEY
EOL

## 4. Set Environment and Verify AWS Configuration
*Set environment once [dev, test, prod] - all deployment cells will use these values*

In [35]:
import os

# Set environment variables (change 'dev' to 'test' or 'prod' as needed)
ENV = input("Enter environment (dev, test, prod): ").strip()
if ENV not in ['dev', 'test', 'prod']:
    raise ValueError("Invalid environment. Choose from 'dev', 'test', or 'prod'.")

os.environ['AWS_PROFILE'] = 'radx-rep'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

print(f'Environment: {ENV}')
print('Testing AWS connectivity...')

!aws sts get-caller-identity
print('✅ AWS configuration verified')

Environment: dev
Testing AWS connectivity...
{
    "UserId": "AIDAZCZPSGGIYLJUSEA2H",
    "Account": "624480629137",
    "Arn": "arn:aws:iam::624480629137:user/yan-local-dev"
}
✅ AWS configuration verified


---
# 🚀 Stack Deployment

**Important**: Wait for each stack to complete before proceeding to the next one. Check the AWS Console CloudFormation page to monitor progress.

## 5. Deploy Networking Stack
*Creates VPC, subnets, security groups, and networking infrastructure*

In [ ]:
print('🔗 Deploying Networking Stack...')

!aws cloudformation deploy \
  --stack-name RADx-Networking-{ENV} \
  --template-file modules/Networking.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ Networking Stack deployment complete')

## 6. Deploy S3 Stack
*Creates S3 buckets for application data, uploads, and artifacts*

In [ ]:
print('🗄️ Deploying S3 Stack...')

!aws cloudformation deploy \
  --stack-name RADx-S3-{ENV} \
  --template-file modules/S3.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ S3 Stack deployment complete')

## 7. Deploy Secrets Manager Stack
*Creates secrets for database credentials and API keys*

In [ ]:
print('🔐 Deploying Secrets Manager Stack...')

!aws cloudformation deploy \
  --stack-name RADx-SecretsManager-{ENV} \
  --template-file modules/SecretsManager.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ Secrets Manager Stack deployment complete')

## 8. Deploy Load Balancer Stack
*Creates Application Load Balancer, target groups, and routing rules*

In [ ]:
print('⚖️ Deploying Load Balancer Stack...')

!aws cloudformation deploy \
  --stack-name RADx-LoadBalancer-{ENV} \
  --template-file modules/LoadBalancer.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ Load Balancer Stack deployment complete')

## 9. Deploy RDS Stack
*Creates RDS database instances and related infrastructure*

In [ ]:
print('🗃️ Deploying RDS Stack... (This may take 10-15 minutes)')

!aws cloudformation deploy \
  --stack-name RADx-RDS-{ENV} \
  --template-file modules/RDS.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ RDS Stack deployment complete')

## 10. Deploy Route53 Stack

*Creates DNS records and routing configurations*

*NOTE: This will be organization dependent, and you will need to change DNS CNAMES, A records, etc. Consult with your hostmaster for your DNS records, and whether this stack is necessary.*

In [ ]:
print('🌐 Deploying Route53 Stack...')

!aws cloudformation deploy \
  --stack-name RADx-Route53-{ENV} \
  --template-file modules/Route53.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ Route53 Stack deployment complete')

## 11. Deploy CloudWatch Stack
*Creates monitoring, logging, and alerting infrastructure*

In [ ]:
print('📊 Deploying CloudWatch Stack...')

!aws cloudformation deploy \
  --stack-name RADx-CloudWatch-{ENV} \
  --template-file modules/CloudWatch.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ CloudWatch Stack deployment complete')

## 12. Deploy SQS Stack
*Creates SQS queues for message processing*

In [ ]:
print('📬 Deploying SQS Stack...')

!aws cloudformation deploy \
  --stack-name RADx-SQS-{ENV} \
  --template-file modules/SQS.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ SQS Stack deployment complete')

## 13. Deploy ECR Stack
*Creates ECR repositories for container images*

In [ ]:
print('📦 Deploying ECR Stack...')

!aws cloudformation deploy \
  --stack-name RADx-ECR-{ENV} \
  --template-file modules/ECR.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ ECR Stack deployment complete')

## 14. Deploy OpenSearch Stack
*Creates the OpenSearch domain and supporting resources*

In [ ]:
print('🔎 Deploying OpenSearch Stack... (This may take 10-20 minutes)')

!aws cloudformation deploy \
  --stack-name RADx-OpenSearch-{ENV} \
  --template-file modules/OpenSearch.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ OpenSearch Stack deployment complete')

🔎 Deploying OpenSearch Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - RADx-OpenSearch-dev
✅ OpenSearch Stack deployment complete


## 15. Deploy ECS Stack
*Creates ECS cluster, services, and container definitions*

In [47]:
print('🐳 Deploying ECS Stack... (This may take 10-20 minutes)')

!aws cloudformation deploy \
  --stack-name RADx-ECS-{ENV} \
  --template-file modules/ECS.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ ECS Stack deployment complete')
print('🎉 All stacks deployed! Your RADx Data Hub is ready.')

🐳 Deploying ECS Stack... (This may take 10-20 minutes)

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - RADx-ECS-dev
✅ ECS Stack deployment complete
🎉 All stacks deployed! Your RADx Data Hub is ready.


## 16. Deploy SageMaker Stack *(pending)*
*Creates SageMaker doamin*

In [ ]:
print('🐳 Deploying SageMaker Stack... (This may take 10-20 minutes)')

!aws cloudformation deploy \
  --stack-name RADx-SageMaker-{ENV} \
  --template-file modules/SageMaker.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ SageMaker Stack deployment complete')

## 17. Deploy Lambda Stack *(pending)*
*Creates Lambda functions for data processing, automation, and integrations*

In [ ]:
print('⚡ Deploying Lambda Stack...')

!aws cloudformation deploy \
  --stack-name RADx-Lambda-{ENV} \
  --template-file modules/Lambda.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ Lambda Stack deployment complete')

⚡ Deploying Lambda Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete


## 18. Deploy EventBridge Stack *(pending)*
*Creates EventBridge rules to trigger Lambda functions on schedules*

In [ ]:
print('⏰ Deploying EventBridge Stack...')

!aws cloudformation deploy \
  --stack-name RADx-EventBridge-{ENV} \
  --template-file modules/EventBridge.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=radx environment={ENV}

print('✅ EventBridge Stack deployment complete')
print('🎉 All stacks deployed! Your RADx Data Hub is ready.')

---
# ✅ Post-Deployment Verification

After all stacks are deployed, verify the installation:

In [ ]:
print('🔍 Checking stack status...')

!aws cloudformation list-stacks \
  --stack-status-filter CREATE_COMPLETE UPDATE_COMPLETE \
  --query 'StackSummaries[?contains(StackName, `RADx`)].{Name:StackName,Status:StackStatus}' \
  --output table
  
print('✅ Stack status check complete')

# 🔧 Troubleshooting

## Common Issues:

### Stack Creation Failed
```python
# Check stack events for errors
!aws cloudformation describe-stack-events --stack-name RADx-{ENV}-[STACK-NAME]
```

### Resource Already Exists
- S3 bucket names must be globally unique
- Modify bucket names in the S3.yaml template

### Permission Denied
- Ensure your IAM user has all required permissions
- Check AWS credentials are correctly configured

### Import Value Not Found
- Ensure previous stacks completed successfully
- Verify stack names match exactly

---
# 🧹 Cleanup

**⚠️ WARNING**: This will delete ALL resources and data. Make sure you have backups of any important data.

To delete all stacks when done testing:

## Check Current Stacks Before Cleanup

In [ ]:
print('📋 Current RADx stacks:')

!aws cloudformation list-stacks \
  --stack-status-filter CREATE_COMPLETE UPDATE_COMPLETE \
  --query 'StackSummaries[?contains(StackName, `RADx`)].{Name:StackName,Status:StackStatus,Created:CreationTime}' \
  --output table

## Manual S3 Bucket Cleanup (Required First)
*S3 buckets with content cannot be deleted by CloudFormation*

In [ ]:
print('🗁️ Emptying S3 buckets before stack deletion...')

# List and empty all RADx S3 buckets
import subprocess

result = subprocess.run(['aws', 's3', 'ls'], capture_output=True, text=True)
buckets = [line.split()[-1] for line in result.stdout.strip().split('\n') if 'radx' in line.lower()]

for bucket in buckets:
    print(f'Emptying bucket: {bucket}')
    !aws s3 rm s3://{bucket} --recursive
    !aws s3api delete-bucket-versioning --bucket {bucket} --versioning-configuration Status=Suspended

print('✅ S3 buckets emptied')

## Delete CloudFormation Stacks
*Stacks must be deleted in reverse order due to dependencies*

In [ ]:
print('🗂️ Deleting stacks in reverse dependency order...')

# Delete stacks in reverse order
stacks = [
    f'RADx-ECS-{ENV}',
    f'RADx-ECR-{ENV}',
    f'RADx-SQS-{ENV}',
    f'RADx-CloudWatch-{ENV}',
    f'RADx-Route53-{ENV}',
    f'RADx-RDS-{ENV}',
    f'RADx-ApplicationLoadBalancer-{ENV}',
    f'RADx-SecretsManager-{ENV}',
    f'RADx-S3-{ENV}',
    f'RADx-Networking-{ENV}'
]

for stack in stacks:
    print(f'🗁️ Deleting stack: {stack}')
    !aws cloudformation delete-stack --stack-name {stack} --region us-east-1
    
    print(f'⏳ Waiting for {stack} deletion to complete...')
    !aws cloudformation wait stack-delete-complete --stack-name {stack} --region us-east-1
    
    print(f'✅ {stack} deleted successfully\n')

print('🎉 Cleanup complete!')

## Verify Cleanup

In [ ]:
print('🔍 Checking for remaining RADx resources...')

print('CloudFormation Stacks:')
!aws cloudformation list-stacks \
  --query 'StackSummaries[?contains(StackName, `RADx`) && StackStatus != `DELETE_COMPLETE`].{Name:StackName,Status:StackStatus}' \
  --output table

print('\nS3 Buckets:')
result = subprocess.run(['aws', 's3', 'ls'], capture_output=True, text=True)
radx_buckets = [line for line in result.stdout.split('\n') if 'radx' in line.lower()]
if radx_buckets:
    for bucket in radx_buckets:
        print(bucket)
else:
    print('No RADx S3 buckets found')

print('\nLoad Balancers:')
!aws elbv2 describe-load-balancers \
  --query 'LoadBalancers[?contains(LoadBalancerName, `RADx`)].LoadBalancerName' \
  --output text || echo 'No RADx load balancers found'

---
# 📚 Additional Information

## Architecture Overview
- **VPC**: Isolated network environment
- **Public Subnets**: ALB and NAT Gateway
- **Private Subnets**: ECS services and RDS
- **Security Groups**: Network access control
- **ECS Fargate**: Serverless container hosting
- **RDS**: Managed database service
- **S3**: Object storage for files and artifacts

## Cost Optimization Tips
- Use `t3.micro` or `t3.small` for development
- Enable S3 lifecycle policies
- Set up CloudWatch billing alerts
- Delete unused resources regularly

## Security Considerations
- Rotate AWS credentials regularly
- Use least privilege IAM policies
- Enable VPC Flow Logs
- Monitor CloudTrail logs

## Support
For issues or questions:
1. Check CloudFormation events in AWS Console
2. Review CloudWatch logs
3. Consult AWS documentation
4. Contact your AWS support team

---
# 📚 Next Steps 


1. **Deploy Containers to ECR**: ...
2. **Obtain Domains and SSL Certificates**: ...
3. **Replace Lambda code with your actual code via S3 or container images.**
